# Processing UNSW flows

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
!ls ../raw/unsw_nb15/data

'ls' is not recognized as an internal or external command,
operable program or batch file.


### Load data

In [3]:
root = Path('../raw/unsw_nb15/data')
features = pd.read_csv(root /'NetFlow_v3_Features.csv')
flows = pd.read_csv(root / 'NF-UNSW-NB15-v3.csv')

In [4]:
non_ordinal = [
    'IPV4_SRC_ADDR',
    'IPV4_DST_ADDR',
    'L4_SRC_PORT',
    'L4_DST_PORT',
    'PROTOCOL',
    'L7_PROTO',
    'ICMP_TYPE',
    'ICMP_IPV4_TYPE',
    'DNS_QUERY_ID',
    'DNS_QUERY_TYPE',
    'DNS_TTL_ANSWER',
    'FTP_COMMAND_RET_CODE',
]

for col in non_ordinal:
    print(
        col, len(pd.unique(flows[col])))

IPV4_SRC_ADDR 40
IPV4_DST_ADDR 40
L4_SRC_PORT 64597
L4_DST_PORT 64617
PROTOCOL 255
L7_PROTO 141
ICMP_TYPE 1188
ICMP_IPV4_TYPE 256
DNS_QUERY_ID 64196
DNS_QUERY_TYPE 20
DNS_TTL_ANSWER 102
FTP_COMMAND_RET_CODE 16


In [5]:
high_count_categorical = [
    'L4_SRC_PORT', 'L4_DST_PORT', 'ICMP_TYPE', 'DNS_QUERY_ID',
    'PROTOCOL', 'L7_PROTO', 'DNS_TTL_ANSWER', 'ICMP_IPV4_TYPE'
]

for c in high_count_categorical:
    non_ordinal.remove(c)
low_count_categorical = non_ordinal

targets = ('Label', 'Attack')

numerical = [c for c in flows.columns 
             if c not in low_count_categorical 
             and c not in high_count_categorical
             and c not in targets]


print(f'numerical: {numerical}')
print(f'categorical: {low_count_categorical}')
print(f'non sparse categorical: {high_count_categorical}')

numerical: ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX', 'SRC_TO_DST_IAT_AVG', 'SRC_TO_DST_IAT_STDDEV', 'DST_TO_SRC_IAT_MIN', 'DST_TO_SRC_IAT_MAX', 'DST_TO_SRC_IAT_AVG', 'DST_TO_SRC_IAT_STDDEV']
categorical: ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'DNS_QUERY_TYPE', 'FTP_COMMAND_RET_CODE']
non

In [6]:
# filter inf values
print(len(flows))
for c in flows.columns:
    flows = flows[flows[c] != np.inf]
print(len(flows))

2365424
2242931


In [7]:
flows = flows.sort_values(by='FLOW_START_MILLISECONDS')

In [8]:
flows.columns

Index(['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR',
       'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO',
       'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS',
       'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS',
       'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT',
       'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN',
       'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES',
       'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS',
       'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS',
       'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT',
       'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES',
       'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES',
       'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT',
       'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE',
       'DNS_TTL_ANSWER', 'FTP_COMMAN

# Flow Graph processing

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder, StandardScaler, LabelEncoder)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            dtype="float32"
        ), low_count_categorical + ['Attack']),
        # ('lab', LabelEncoder(), non_sparse_categorical),
        ("num", StandardScaler(), numerical),
    ]
)

In [11]:
flows.columns

Index(['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR',
       'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO',
       'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS',
       'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS',
       'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT',
       'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN',
       'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES',
       'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS',
       'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS',
       'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT',
       'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES',
       'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES',
       'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT',
       'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE',
       'DNS_TTL_ANSWER', 'FTP_COMMAN

In [ ]:
flows.drop('Label', axis=1, inplace=True) # inferred from Attack

In [15]:
flows['src'] = flows['IPV4_SRC_ADDR'].astype(str) + ':' + flows['L4_SRC_PORT'].astype(str)
flows['dst'] = flows['IPV4_DST_ADDR'].astype(str) + ':' + flows['L4_DST_PORT'].astype(str)
flows.drop(['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT'], axis=1, inplace=True)

In [21]:
np.unique(flows.Attack)

array(['Analysis', 'Backdoor', 'Benign', 'DoS', 'Exploits', 'Fuzzers',
       'Generic', 'Reconnaissance', 'Shellcode', 'Worms'], dtype=object)

In [ ]:
flows['Attack'] = flows['Attack'] != 'Benign' # only run once

1230072    False
1230073    False
1230074    False
1230075     True
1230076    False
           ...  
410789     False
410790     False
410791     False
410792      True
410793     False
Name: Attack, Length: 2242931, dtype: bool

### Torch Graph Encoding

In [32]:
import torch
import numpy as np
import torch
import networkx as nx
from tqdm import tqdm
from torch_geometric.data import Data
from torch_geometric.transforms import LineGraph

def graph_encode(data, edge_cols: list, linegraph: bool, target_col: str):
    # ----- build edge features -----
    attrs = [c for c in data.columns if c not in edge_cols + [target_col]]

    x = data[attrs].to_numpy(dtype=np.float32)
    edge_attr = torch.tensor(x, dtype=torch.float)

    # ----- edge labels -----
    edge_y = torch.tensor(
        data[target_col].values, dtype=torch.long
    )

    # ----- node indexing -----
    nodes = pd.concat([data['src'], data['dst']]).unique()
    node_map = {n: i for i, n in enumerate(nodes)}

    src_name, dst_name = edge_cols
    src = data[src_name].map(node_map).to_numpy()
    dst = data[dst_name].map(node_map).to_numpy()

    edge_index = torch.tensor(
        np.stack([src, dst]), dtype=torch.long
    )

    # ----- PyG graph -----
    G = Data(
        edge_index=edge_index,
        edge_attr=edge_attr,
        edge_y=edge_y,
        num_nodes=len(nodes)
    )

    # ----- optional line graph -----
    if linegraph:
        G = LineGraph()(G)

    return G, node_map


_ = graph_encode(flows, ['src', 'dst'], False, target_col='Attack')


In [33]:
n_flows = len(flows)
size = 500
for i, start in enumerate(
    tqdm(range(0, n_flows - 1, size))
):
    window_flows = flows.iloc[start:start + size]
    window_graph, node_map = graph_encode(window_flows, 
                       linegraph=False, 
                       edge_cols=['src', 'dst'], 
                       target_col='Attack')
    

# look at the last one
print(f'graph {i}/{len(flows) // 500}')
print(
    f'graph statistics: '
    f'edges = {window_graph.num_edges}, '
    f'nodes = {window_graph.num_nodes}'
)


100%|██████████| 4486/4486 [00:10<00:00, 433.43it/s]

graph 4485/4485
graph statistics: edges = 431, nodes = 494


In [34]:
flows.to_csv('../interm/unsw_nb15_processed.csv')

OSError: Cannot save file into a non-existent directory: '..\interm'

### Old DGL method

In [20]:
from sklearn.preprocessing import (LabelEncoder)
import networkx as nx
import dgl
from tqdm import tqdm

def graph_encode(data, edge_cols: list, linegraph: bool, target_col: str):
    attrs = [c for c in data.columns if c not in edge_cols + [target_col]]
    data['x'] = data[attrs].values.tolist()
    data['x'] = data['x'].apply(lambda x: np.array(x, dtype=np.float32))

    G = nx.from_pandas_edgelist(
            data, source='src', 
            target='dst', 
            edge_attr=['x', 'Attack'], 
            create_using=nx.MultiGraph())
    
    G = G.to_directed()
    G = dgl.from_networkx(G, edge_attrs=['x', 'Attack'])
    if linegraph:
        G = G.line_graph(shared=True)
        
    print(G.number_of_nodes(), G.number_of_edges())
    return G

def yield_windows(processed_flows, size, linegraph=False, **kwargs):
    n_flows = len(processed_flows)
    for start in tqdm(
         range(0, n_flows-1, size)
    ):
        window_flows = processed_flows.iloc[start:start+size]
        yield graph_encode(window_flows, **kwargs)


for i, window_graph in enumerate(
    yield_windows(flows, 
                size=500, 
                linegraph=False,
                edge_cols=('src', 'dst'),
                target_col='Attack')
):
    print(f'graph {i}/{len(flows // 500)}')
    print(f'graph statistics: edges = {window_graph.num_edges()}, nodes = {window_graph.num_nodes()}')
    del window_graph
        


ModuleNotFoundError: No module named 'distutils'